<a href="https://colab.research.google.com/github/ghduf0201-oss/GPT2.0-0toHero/blob/main/notebook_03_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — MLP on Tiny Shakespeare

이제 모델은 2번 MLP를 그대로 두고, **데이터만 `tiny Shakespeare`로 바꿉니다.**

문제는 여전히 **fixed context -> next char** 입니다.

In [3]:
# 1. 코랩 환경에 PDF 텍스트 수색용 라이브러리 강제 탑재
!pip install -q PyPDF2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import PyPDF2
import requests

# =====================================================================
# 🎯 [사령관 오더 전선] 여기에 사령관이 원하는 PDF 주소 링크만 붙여넣어!
# =====================================================================
pdf_url = "https://www.federalreserve.gov/mediacenter/files/FOMCpresconf20260617.pdf"  # 👈 이 링크를 지우고 사령관의 PDF 링크를 넣으렴!
# =====================================================================

extracted_text = ""

# 인터넷 링크(URL) 다운로드 및 임시 저장 전술
if pdf_url and pdf_url.startswith("http"):
    print("🚀 인터넷에서 PDF 파일 추적 및 다운로드 중...")
    response = requests.get(pdf_url)
    with open("temp_downloaded.pdf", "wb") as f:
        f.write(response.content)
    pdf_file_to_read = "temp_downloaded.pdf"
else:
    print("❌ 유효한 HTTP 링크가 아닙니다. pdf_url을 다시 확인해 주십시오.")

# PDF 내부의 글자 분자들만 탈탈 털어내는 세탁 공정
print(f"📄 '{pdf_file_to_read}' 자산에서 순수 텍스트 추출 가동...")
with open(pdf_file_to_read, "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)
    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        page_text = page.extract_text()
        if page_text:
            extracted_text += page_text + "\n"

# 카파시 모델의 인풋 텍스트 장부로 다이렉트 상환
text = extracted_text

# [기존 카파시 코드 연동 영역] - 데이터셋 변동에 따른 어휘 사전 자동 리밸런싱
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("\n=== 📈 최종 데이터 장부 정산 결과 ===")
print("텍스트 총 글자수 (text length):", len(text))
print("어휘 사전 크기 (vocab_size):", vocab_size)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.2 MB/s eta 0:00:00
🚀 인터넷에서 PDF 파일 추적 및 다운로드 중...
📄 'temp_downloaded.pdf' 자산에서 순수 텍스트 추출 가동...

=== 📈 최종 데이터 장부 정산 결과 ===
텍스트 총 글자수 (text length): 41367
어휘 사전 크기 (vocab_size): 77


## 1. Sliding-window dataset

In [4]:
class CharSequenceNextCharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + self.block_size]
        return x, y

block_size = 16
dataset = CharSequenceNextCharDataset(data, block_size)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)
print("decoded x:", ''.join(itos[i.item()] for i in xb[0]))
print("decoded y:", itos[yb[0].item()])

xb.shape: torch.Size([128, 16])
yb.shape: torch.Size([128])
decoded x: e was the normal
decoded y:  


## 2. MLP model

In [5]:
class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([128, 77])
initial loss: 4.354556560516357


## 3. 학습

In [6]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(10):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.4343
epoch  1 | train loss 1.8920
epoch  2 | train loss 1.6435
epoch  3 | train loss 1.4502
epoch  4 | train loss 1.2900
epoch  5 | train loss 1.1443
epoch  6 | train loss 1.0095
epoch  7 | train loss 0.8876
epoch  8 | train loss 0.7812
epoch  9 | train loss 0.6865


## 4. Sampling

In [14]:
@torch.no_grad()
def sample_mlp(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=300):
    model.eval()
    context = [0] * block_size
    for ch in start_text:
        if ch in stoi:
            context = context[1:] + [stoi[ch]]
    out = list(start_text)
    for _ in range(max_new_tokens):
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1).item()
        out.append(itos[ix])
        context = context[1:] + [ix]
    return "".join(out)

print(sample_mlp(model, block_size, stoi, itos, device, start_text="Chairman Warsh:", max_new_tokens=400))

Chairman Warsh:s Press Cougrestro to 
thispercen is that. I than whele saye polickss, dos sechave a condict insom ore 
inul goan . I thos comon ad reveryt relyour 
ours. Fed blok as ande cous and yith midin the what delleat we dole this has 
thingseaty in erest prive to the tome of thise compancr ibferst. Bughistion ormetter on that there in like for the puraurag exterts. But we biltorests the promus.  an the co


## 5. 정리

- 같은 MLP 모델을 더 큰 텍스트에도 적용할 수 있습니다.
- 바뀌는 것은 dataset입니다.
- 하지만 fixed context MLP는 긴 문맥을 잘 다루지 못합니다.